In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
import h5py
from kNN import CustomVectorizedKNeighborsClassifier, CustomVectorizedSortedKNeighborsClassifier, DistanceCalculator

import time

In [3]:
with h5py.File("../data/train_data.h5", "r") as f:
    X_train = f["X"][:]
    y_train = f["y"][:]

with h5py.File("../data/test_data.h5", "r") as f:
    X_test = f["X"][:]
    y_test = f["y"][:]

In [4]:
# Fit the models
n_neighbours = 3

model_sk = KNeighborsClassifier(n_neighbors=n_neighbours)
model_sk.fit(X_train, y_train)

model_custom = CustomVectorizedKNeighborsClassifier(n_neighbors=n_neighbours)
model_custom.fit(X_train, y_train)

model_custom_sort = CustomVectorizedSortedKNeighborsClassifier(n_neighbors=n_neighbours)
model_custom_sort.fit(X_train, y_train)

In [5]:
%%timeit # 548 μs
y_pred_sk = model_sk.predict(X_test)

547 μs ± 4.55 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [6]:
%%timeit # 635 μs 
y_pred_custom = model_custom.predict(X_test, DistanceCalculator.Method.ThreeDBroadcasting)

612 μs ± 2.05 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [7]:
%%timeit # 635 μs 
y_pred_custom_cp = model_custom.predict(X_test, DistanceCalculator.Method.CrossProduct)

403 μs ± 8.27 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [8]:
%%timeit # 632 μs
y_pred_custom_sort = model_custom_sort.predict(X_test, DistanceCalculator.Method.ThreeDBroadcasting)    

613 μs ± 3.76 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [9]:
%%timeit # 632 μs
y_pred_custom_sort_cp = model_custom_sort.predict(X_test, DistanceCalculator.Method.CrossProduct)    

408 μs ± 14.1 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [10]:
y_pred_sk = model_sk.predict(X_test)
y_pred_custom = model_custom.predict(X_test)
y_pred_custom_cp = model_custom.predict(X_test, DistanceCalculator.Method.CrossProduct)
y_pred_custom_sort = model_custom_sort.predict(X_test)    
y_pred_custom_sort_cp = model_custom_sort.predict(X_test, DistanceCalculator.Method.CrossProduct)

accuracy_sk = accuracy_score(y_pred_sk, y_test)
print(f"Accuracy (sk): {accuracy_sk:%}")

accuracy_custom = accuracy_score(y_pred_custom, y_test)
print(f"Accuracy (custom): {accuracy_custom:%}")

accuracy_custom_cp = accuracy_score(y_pred_custom_cp, y_test)
print(f"Accuracy (custom, cross product): {accuracy_custom_cp:%}")

accuracy_custom_sort = accuracy_score(y_pred_custom_sort, y_test)
print(f"Accuracy (custom sorted): {accuracy_custom_sort:%}")

accuracy_custom_sort_cp = accuracy_score(y_pred_custom_sort_cp, y_test)
print(f"Accuracy (custom, cross product): {accuracy_custom_sort_cp:%}")

Accuracy (sk): 80.612245%
Accuracy (custom): 78.571429%
Accuracy (custom, cross product): 78.571429%
Accuracy (custom sorted): 78.571429%
Accuracy (custom, cross product): 78.571429%


In [12]:
def train_and_fit_sk(n_neighbours):
    model = KNeighborsClassifier(n_neighbors=n_neighbours)
    model.fit(X_train, y_train)
    return model.predict(X_test)

def train_and_fit_custom(n_neighbours):
    model = CustomVectorizedKNeighborsClassifier(n_neighbors=n_neighbours)
    model.fit(X_train, y_train)
    return model.predict(X_test)

def train_and_fit_custom_sort(n_neighbours):
    model = CustomVectorizedSortedKNeighborsClassifier(n_neighbors=n_neighbours)
    model.fit(X_train, y_train)
    return model.predict(X_test)

K = 10
label_width = 25

start = time.perf_counter()
y_pred_multi_sk = np.array([train_and_fit_sk(n_neighbours) for n_neighbours in range(1,K)])
end = time.perf_counter()
print(f"{'Elapsed (sk):':<{label_width}} {end - start:.6f} seconds")

start = time.perf_counter()
y_pred_multi_custom = np.array([train_and_fit_custom(n_neighbours) for n_neighbours in range(1,K)])
end = time.perf_counter()
print(f"{'Elapsed (Custom):':<{label_width}} {end - start:.6f} seconds")

start = time.perf_counter()
y_pred_multi_custom_sort = np.array([train_and_fit_custom_sort(n_neighbours) for n_neighbours in range(1,K)])
end = time.perf_counter()
print(f"{'Elapsed (Custom Sort):':<{label_width}} {end - start:.6f} seconds")

Elapsed (sk):             0.010807 seconds
Elapsed (Custom):         0.005545 seconds
Elapsed (Custom Sort):    0.004645 seconds
